### **This new notebook is for the 4 Features, 1 qubit per feature experiment, in total we have 4 Qubits. This notebook reutilizes a lot of code in QAOA_4F2Q_8Q.ipynb if you want to get a deeper insight you can refer to that notebook is well documented code**

**Imports**

In [9]:
import math
from typing import Any

import kirin
from kirin.dialects import ilist
from bloqade import qasm2

**Variables for 4F1Q 4 Qubit in total experiment**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from itertools import product

# ========= Problem definition (4 assets / 4 qubits) =========
n_qubits = 4

asset_labels = [
    "A017 (Gov Bonds)",
    "A026 (IG Credit)",
    "A007 (Cash)",
    "A047 (HY Credit)",
]

# Ground truth from brute force
ground_truth_bits = "0011"
ground_truth_x = np.array([0, 0, 1, 1], dtype=int)
ground_truth_energy = -0.007414

# Ising local fields h
h_terms = np.array([
    -0.186803,   # A017
    -0.136982,   # A026
    -0.202042,   # A007
    -0.078664,   # A047
], dtype=float)

# Ising couplings J as symmetric matrix
J_terms = np.zeros((n_qubits, n_qubits), dtype=float)
J_terms[0,1] = J_terms[1,0] = 0.060001
J_terms[0,2] = J_terms[2,0] = 0.125001
J_terms[0,3] = J_terms[3,0] = 0.030001
J_terms[1,2] = J_terms[2,1] = 0.075000
J_terms[1,3] = J_terms[3,1] = 0.018001
J_terms[2,3] = J_terms[3,2] = 0.037501

# Constant shift (does not affect argmin, but useful for reporting)
ising_const = 0.597754

print("n_qubits =", n_qubits)
print("ground_truth_bits =", ground_truth_bits)
print("ground_truth_energy =", ground_truth_energy)
print("h_terms =", h_terms)
print("J_terms =\n", J_terms)

n_qubits = 4
ground_truth_bits = 0011
ground_truth_energy = -0.007414
h_terms = [-0.186803 -0.136982 -0.202042 -0.078664]
J_terms =
 [[0.       0.060001 0.125001 0.030001]
 [0.060001 0.       0.075    0.018001]
 [0.125001 0.075    0.       0.037501]
 [0.030001 0.018001 0.037501 0.      ]]


**Calculate energy function**

In [2]:
def bitstring_to_spin_array(bitstring: str) -> np.ndarray:
    # bit 0 -> s=-1, bit 1 -> s=+1
    return np.array([1 if b == "1" else -1 for b in bitstring], dtype=float)

def ising_energy_from_bitstring(bitstring: str, h_terms, J_terms, const=0.0) -> float:
    s = bitstring_to_spin_array(bitstring)
    e = 0.0

    # pair terms
    n = len(s)
    for i in range(n):
        for j in range(i + 1, n):
            e += J_terms[i, j] * s[i] * s[j]

    # local fields
    e += np.dot(h_terms, s)

    return float(e + const)

# quick sanity check
all_bitstrings = ["".join(bits) for bits in product("01", repeat=n_qubits)]
energy_table = pd.DataFrame({
    "bitstring": all_bitstrings,
    "energy": [ising_energy_from_bitstring(bs, h_terms, J_terms, ising_const) for bs in all_bitstrings]
}).sort_values("energy", ascending=True).reset_index(drop=True)

energy_table.head(10)

,bitstring,energy
0,1101,0.267848
1,0111,0.282368
2,1011,0.306728
3,1110,0.325090
4,1111,0.338768
5,1010,0.365054
6,0110,0.388694
7,1100,0.404174
8,0011,0.490332
9,1001,0.535808


**Helper functions to analyze the data**

In [5]:
def normalize_prob_dict(prob_dict):
    total = sum(prob_dict.values())
    if total == 0:
        return prob_dict
    return {k: v / total for k, v in prob_dict.items()}

def expected_energy_from_probs(prob_dict, h_terms, J_terms, const=0.0):
    prob_dict = normalize_prob_dict(prob_dict)
    e = 0.0
    for bitstring, p in prob_dict.items():
        e += p * ising_energy_from_bitstring(bitstring, h_terms, J_terms, const)
    return float(e)

def ground_truth_probability(prob_dict, ground_truth_bits):
    prob_dict = normalize_prob_dict(prob_dict)
    return float(prob_dict.get(ground_truth_bits, 0.0))

def marginal_one_probs(prob_dict, n_qubits):
    prob_dict = normalize_prob_dict(prob_dict)
    marginals = np.zeros(n_qubits, dtype=float)

    for bitstring, p in prob_dict.items():
        for i, b in enumerate(bitstring):
            if b == "1":
                marginals[i] += p

    return marginals

def top_states_from_probs(prob_dict, top_k=10):
    prob_dict = normalize_prob_dict(prob_dict)
    rows = []
    for bitstring, p in prob_dict.items():
        rows.append({
            "bitstring": bitstring,
            "probability": p,
            "energy": ising_energy_from_bitstring(bitstring, h_terms, J_terms, ising_const)
        })
    df = pd.DataFrame(rows).sort_values("probability", ascending=False).reset_index(drop=True)
    return df.head(top_k)

**convert ising data to format expected**

In [6]:
# Convert current Ising data to the format expected by build_qaoa_ising_kernel

h_terms_list = [(i, float(h_terms[i])) for i in range(n_qubits)]

J_terms_list = []
for i in range(n_qubits):
    for j in range(i + 1, n_qubits):
        if abs(J_terms[i, j]) > 1e-12:
            J_terms_list.append((i, j, float(J_terms[i, j])))

print("h_terms_list =", h_terms_list)
print("J_terms_list =", J_terms_list)

h_terms_list = [(0, -0.186803), (1, -0.136982), (2, -0.202042), (3, -0.078664)]
J_terms_list = [(0, 1, 0.060001), (0, 2, 0.125001), (0, 3, 0.030001), (1, 2, 0.075), (1, 3, 0.018001), (2, 3, 0.037501)]


### **Construct the circuit**

In [7]:
def build_qaoa_ising_kernel(n_qubits, h_terms, J_terms):
    @qasm2.extended
    def kernel(gamma: ilist.IList[float, Any], beta: ilist.IList[float, Any]):
        q = qasm2.qreg(n_qubits)

        for i in range(n_qubits):
            qasm2.h(q[i])

        for layer in range(len(gamma)):
            g = gamma[layer]
            b = beta[layer]

            for k in range(len(h_terms)):
                term = h_terms[k]
                i = term[0]
                hi = term[1]
                qasm2.rz(q[i], 2.0 * g * hi)

            for k in range(len(J_terms)):
                term = J_terms[k]
                i = term[0]
                j = term[1]
                Jij = term[2]

                qasm2.cx(q[i], q[j])
                qasm2.rz(q[j], 2.0 * g * Jij)
                qasm2.cx(q[i], q[j])

            for i in range(n_qubits):
                qasm2.rx(q[i], 2.0 * b)

        return q

    return kernel

In [10]:
h_terms_list = [(i, float(h_terms[i])) for i in range(n_qubits)]

J_terms_list = []
for i in range(n_qubits):
    for j in range(i + 1, n_qubits):
        if abs(J_terms[i, j]) > 1e-12:
            J_terms_list.append((i, j, float(J_terms[i, j])))

qaoa_kernel = build_qaoa_ising_kernel(
    n_qubits=n_qubits,
    h_terms=h_terms_list,
    J_terms=J_terms_list
)

print("QAOA kernel construido.")

QAOA kernel construido.


**Summary Table Function**

In [ ]:
summary_df = pd.DataFrame([
    {
        "P": r["P"],
        "runtime_sec": r["runtime_sec"],
        "best_fun": r["best_fun"],
        "expected_energy": r["expected_energy"],
        "ground_truth_probability": r["ground_truth_probability"],
    }
    for r in experiment_results
])

summary_df.sort_values("P").reset_index(drop=True)

**1ST Graph, Expected energy vs P**

In [ ]:
plot_df = summary_df.sort_values("P")

plt.figure(figsize=(7,5))
plt.plot(plot_df["P"], plot_df["expected_energy"], marker="o")
plt.axhline(ground_truth_energy, linestyle="--")
plt.xlabel("QAOA depth P")
plt.ylabel("Expected energy")
plt.title("Best expected energy vs P (4 assets / 4 qubits)")
plt.grid(True, alpha=0.3)
plt.show()

**2ST Graph, Runtime vs P**

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(plot_df["P"], plot_df["runtime_sec"], marker="o")
plt.xlabel("QAOA depth P")
plt.ylabel("Runtime (sec)")
plt.title("Runtime vs P (4 assets / 4 qubits)")
plt.grid(True, alpha=0.3)
plt.show()

**3RD Graph, Ground Truth probability VS P**

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(plot_df["P"], plot_df["ground_truth_probability"], marker="o")
plt.xlabel("QAOA depth P")
plt.ylabel(f'Probability of ground truth state {ground_truth_bits}')
plt.title("Ground-truth probability vs P (4 assets / 4 qubits)")
plt.grid(True, alpha=0.3)
plt.show()

**Best run and top states table**

In [ ]:
best_idx = summary_df["ground_truth_probability"].idxmax()
best_run = experiment_results[best_idx]

print("Best run by ground-truth probability:")
print("P =", best_run["P"])
print("best_gamma =", best_run["best_gamma"])
print("best_beta =", best_run["best_beta"])
print("expected_energy =", best_run["expected_energy"])
print("ground_truth_probability =", best_run["ground_truth_probability"])

top_states_from_probs(best_run["probs_opt"], top_k=10)

**1ST Qubit individual probability graph**

In [ ]:
marginals = marginal_one_probs(best_run["probs_opt"], n_qubits)

plt.figure(figsize=(8,5))
plt.bar(range(n_qubits), marginals)
plt.xticks(range(n_qubits), asset_labels, rotation=25, ha="right")
plt.ylabel("P(qubit = 1)")
plt.title(f"Marginal probability of measuring 1 (best run, P={best_run['P']})")
plt.tight_layout()
plt.show()

**Graph 5:**

In [ ]:
top_df = top_states_from_probs(best_run["probs_opt"], top_k=16).sort_values("probability", ascending=False)

plt.figure(figsize=(10,5))
plt.bar(top_df["bitstring"], top_df["probability"])
plt.xlabel("Bitstring")
plt.ylabel("Probability")
plt.title(f"Measured state distribution (best run, P={best_run['P']})")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()